# demean

> 对应代码：`EconometricsCode/code_in_notes/Chap.8/demean.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们读入城市数据，只保留 2011 年的截面，并生成污染对数和人均 GDP 对数两个变量。

In [ ]:
%%stata
clear
use ../datasets/citydata
keep if Year==2011
gen log_pollu=log(v268)
gen log_pgdp=log(v84)-log(v4)

我们把对数人均 GDP 的样本均值存入局部宏 m_pgdp，稍后用它在均值处计算边际效应。

In [ ]:
%%stata
su log_pgdp
local m_pgdp=r(mean)

我们生成两个版本的平方项：一个是原始平方项，另一个是离差的平方（去平均的平方项），用来对比两种参数化方式。

In [ ]:
%%stata
// 不去平均的平方项
gen log_pgdp2=log_pgdp^2
// 产生去平均的平方项
egen mean_log_pdgp=mean(log_pgdp)
gen dm_log_pgdp2=(log_pgdp-mean_log_pdgp)^2

我们先用原始平方项回归污染对数，r 选项报告稳健标准误。在含平方项的模型中，一次项系数是在 log_pgdp=0 处的边际效应，经济意义不明显。

In [ ]:
%%stata
// 回归
reg log_pollu log_pgdp log_pgdp2, r

于是我们在样本均值处手工计算边际效应：对 log_pgdp 求导得 β1+2β2·log_pgdp，代入均值即得平均经济发展水平附近的边际效应。

In [ ]:
%%stata
di _b[log_pgdp]+_b[log_pgdp2]*2*`m_pgdp'

改用去平均的平方项重新回归，此时一次项系数恰好就是上一步手工算出的、样本均值处的边际效应，二者数值完全一致，说明去平均只是重新参数化，并不改变模型本身。

In [ ]:
%%stata
reg log_pollu log_pgdp dm_log_pgdp2, r